# 🧩 Day 03a: Subqueries & CTEs — Scalar, Correlated, EXISTS, WITH

---

## 🎯 What You'll Master Today
- Scalar, row, and table subqueries
- Correlated subqueries and EXISTS
- Common Table Expressions (WITH)
- Recursive CTEs for hierarchies

---

## 🏗️ The Russian Doll Analogy

> A subquery is a query inside a query — like Russian nesting dolls.
> A CTE is unpacking those dolls side-by-side so you can see each one clearly.

```
╔══════════════════════════════════════════════════════════════════╗
║                SUBQUERY TYPES CHEAT SHEET                       ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  Scalar Subquery   → returns 1 value     (use in SELECT/WHERE)  ║
║  Row Subquery      → returns 1 row       (use with ROW())       ║
║  Table Subquery    → returns a table      (use in FROM/IN)       ║
║  Correlated        → references outer     (runs per outer row)   ║
║  EXISTS            → checks if rows exist (TRUE/FALSE)           ║
║                                                                  ║
║  CTE (WITH)        → named temp result   (readable subquery)    ║
║  Recursive CTE     → self-referencing    (trees, hierarchies)   ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# Setup
# ============================================

employees = [
    {"id": 1, "name": "Alice",   "dept": "Engineering", "salary": 95000, "mgr_id": None},
    {"id": 2, "name": "Bob",     "dept": "Engineering", "salary": 85000, "mgr_id": 1},
    {"id": 3, "name": "Charlie", "dept": "Marketing",   "salary": 72000, "mgr_id": 1},
    {"id": 4, "name": "Diana",   "dept": "Marketing",   "salary": 68000, "mgr_id": 3},
    {"id": 5, "name": "Eve",     "dept": "Engineering", "salary": 92000, "mgr_id": 1},
    {"id": 6, "name": "Frank",   "dept": "Sales",       "salary": 65000, "mgr_id": None},
    {"id": 7, "name": "Grace",   "dept": "Sales",       "salary": 71000, "mgr_id": 6},
    {"id": 8, "name": "Hank",    "dept": "Engineering", "salary": 98000, "mgr_id": 1},
]

def print_table(rows, columns=None):
    if not rows:
        print("  (empty)")
        return
    if columns is None:
        columns = list(rows[0].keys())
    widths = {c: max(len(str(c)), max(len(str(r.get(c,''))) for r in rows)) for c in columns}
    header = "  " + " | ".join(f"{c:<{widths[c]}}" for c in columns)
    print(header)
    print("  " + "-+-".join("-"*widths[c] for c in columns))
    for r in rows:
        print("  " + " | ".join(f"{str(r.get(c,'')):<{widths[c]}}" for c in columns))

In [ ]:
# ============================================
# 1. Scalar Subquery — returns single value
# ============================================

# SQL: SELECT name, salary, salary - (SELECT AVG(salary) FROM employees) AS diff
#      FROM employees ORDER BY diff DESC;

avg_salary = sum(e["salary"] for e in employees) / len(employees)
print(f"  Scalar subquery: AVG(salary) = {avg_salary:.0f}")
print(f"  SQL: SELECT name, salary - (SELECT AVG(salary) FROM employees) AS diff\n")

result = [{"name": e["name"], "salary": e["salary"], "diff_from_avg": round(e["salary"] - avg_salary)}
          for e in employees]
result.sort(key=lambda x: x["diff_from_avg"], reverse=True)
print_table(result)

In [ ]:
# ============================================
# 2. Table Subquery — IN / NOT IN
# ============================================

# SQL: SELECT * FROM employees
#      WHERE dept IN (SELECT dept FROM employees GROUP BY dept HAVING COUNT(*) >= 3);

print("  Table subquery: employees in departments with 3+ people")
from collections import Counter
dept_counts = Counter(e["dept"] for e in employees)
big_depts = {d for d, c in dept_counts.items() if c >= 3}
print(f"  Subquery result (depts ≥ 3): {big_depts}\n")

result = [e for e in employees if e["dept"] in big_depts]
print_table(result)
print(f"\n  ⚠️ NOT IN gotcha: if subquery returns NULL, NOT IN returns empty set!")
print(f"  Fix: use NOT EXISTS or add WHERE col IS NOT NULL in subquery")

In [ ]:
# ============================================
# 3. Correlated Subquery — runs per outer row
# ============================================

# SQL: SELECT name, salary, dept FROM employees e1
#      WHERE salary > (SELECT AVG(salary) FROM employees e2 WHERE e2.dept = e1.dept);

print("  Correlated subquery: employees earning above their dept average\n")

from collections import defaultdict
dept_salaries = defaultdict(list)
for e in employees:
    dept_salaries[e["dept"]].append(e["salary"])

dept_avg = {d: sum(s)/len(s) for d, s in dept_salaries.items()}
print("  Department averages:")
for d, a in dept_avg.items():
    print(f"    {d}: {a:.0f}")

result = []
for e in employees:
    if e["salary"] > dept_avg[e["dept"]]:
        result.append({"name": e["name"], "dept": e["dept"], "salary": e["salary"],
                       "dept_avg": round(dept_avg[e["dept"]])})
print("\n  Above-average earners:")
print_table(result)
print("\n  💡 Correlated = inner query depends on outer row → slower (N×M potential)")

In [ ]:
# ============================================
# 4. EXISTS vs IN
# ============================================

# SQL: SELECT name FROM employees e
#      WHERE EXISTS (SELECT 1 FROM employees m WHERE m.mgr_id = e.id);

print("  EXISTS: Find employees who are managers of someone\n")

mgr_ids = {e["mgr_id"] for e in employees if e["mgr_id"] is not None}
result = [{"name": e["name"], "id": e["id"]} for e in employees if e["id"] in mgr_ids]
print_table(result)

print("""
  ┌────────────────────────────────────────────────────────────────┐
  │  EXISTS vs IN — When to use which?                            │
  ├────────────────────────────────────────────────────────────────┤
  │  EXISTS                           │  IN                       │
  │  • Stops at first match (fast!)  │  • Loads all subq results │
  │  • Good: large subquery result   │  • Good: small subq result│
  │  • Handles NULLs correctly       │  • NOT IN fails with NULL │
  │  • Correlated (runs per row)     │  • Non-correlated          │
  └────────────────────────────────────────────────────────────────┘
""")

In [ ]:
# ============================================
# 5. CTE (WITH) — readable named subqueries
# ============================================

# SQL:
# WITH dept_stats AS (
#     SELECT dept, AVG(salary) AS avg_sal, COUNT(*) AS cnt
#     FROM employees GROUP BY dept
# ),
# above_avg AS (
#     SELECT e.name, e.salary, ds.avg_sal
#     FROM employees e JOIN dept_stats ds ON e.dept = ds.dept
#     WHERE e.salary > ds.avg_sal
# )
# SELECT * FROM above_avg;

print("  CTE: Chain of named result sets\n")

# CTE 1: dept_stats
dept_stats = []
for dept, sals in dept_salaries.items():
    dept_stats.append({"dept": dept, "avg_sal": round(sum(sals)/len(sals)), "count": len(sals)})
print("  CTE #1 (dept_stats):")
print_table(dept_stats)

# CTE 2: above_avg (uses CTE 1)
d_avg = {ds["dept"]: ds["avg_sal"] for ds in dept_stats}
above_avg = [{"name": e["name"], "salary": e["salary"], "dept_avg": d_avg[e["dept"]]}
             for e in employees if e["salary"] > d_avg[e["dept"]]]
print("\n  CTE #2 (above_avg using dept_stats):")
print_table(above_avg)

print("\n  💡 CTEs make complex queries readable. Can reference earlier CTEs!")

In [ ]:
# ============================================
# 6. Recursive CTE — org hierarchy
# ============================================

# SQL:
# WITH RECURSIVE org_tree AS (
#     SELECT id, name, mgr_id, 0 AS level
#     FROM employees WHERE mgr_id IS NULL
#     UNION ALL
#     SELECT e.id, e.name, e.mgr_id, ot.level + 1
#     FROM employees e JOIN org_tree ot ON e.mgr_id = ot.id
# )
# SELECT * FROM org_tree ORDER BY level, name;

print("  Recursive CTE: Build org tree from employee→manager\n")

emp_lookup = {e["id"]: e for e in employees}

def build_org_tree(employees):
    # Base case: employees with no manager
    queue = [(e, 0) for e in employees if e["mgr_id"] is None]
    result = []
    visited = set()
    
    while queue:
        emp, level = queue.pop(0)
        if emp["id"] in visited:
            continue
        visited.add(emp["id"])
        indent = "  " * level
        result.append({"name": f"{indent}{emp['name']}", "level": level,
                       "manager": emp_lookup[emp['mgr_id']]['name'] if emp['mgr_id'] else '(root)'})
        # Recursive: find direct reports
        for e in employees:
            if e["mgr_id"] == emp["id"]:
                queue.append((e, level + 1))
    return result

tree = build_org_tree(employees)
print_table(tree)
print("\n  💡 Recursive CTE = base case UNION ALL recursive step (with anchor)")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Subquery vs JOIN? | Subquery: filter/compute. JOIN: combine tables. JOIN often faster |
| 2 | Correlated vs non-correlated? | Correlated: references outer query, runs per row. Non-correlated: runs once |
| 3 | EXISTS vs IN? | EXISTS: stops at first match, NULL-safe. IN: loads all, fails with NULL |
| 4 | What is a CTE? | Named temporary result set (WITH clause). More readable than nested subqueries |
| 5 | Recursive CTE use cases? | Org charts, bill of materials, graph traversal, hierarchical data |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Scalar subquery = 1 value. Table subquery = rows      │
## │  • Correlated subquery runs per outer row (slower!)      │
## │  • EXISTS > IN when subquery is large or has NULLs       │
## │  • CTE = readable subquery with WITH keyword             │
## │  • Recursive CTE: anchor + UNION ALL + recursive step    │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Window Functions** — ROW_NUMBER, RANK, LAG, LEAD, PARTITION BY